## Notebook 07 — Baseline LightGBM Model

---

This notebook:
1. Loads + merges data, drops >80% missing columns, removes `TransactionID`
2. Performs stratified 80/20 train-validation split
3. Trains baseline LightGBM classifier with early stopping
4. Evaluates on validation set (ROC-AUC, PR-AUC)
5. Displays best iteration and top 20 feature importances

> **No preprocessing, imputation, or encoding is performed.**

---
## 📦 Step 0 — Import Libraries, Load & Split Data

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

DATA_PATH = '../Datasets/'
TARGET    = 'isFraud'

# Load & merge
print('Loading and merging...')
train_transaction = pd.read_csv(DATA_PATH + 'train_transaction.csv')
train_identity    = pd.read_csv(DATA_PATH + 'train_identity.csv')
train_raw = pd.merge(train_transaction, train_identity, on='TransactionID', how='left')
del train_transaction, train_identity

# Drop >80% missing columns
missing_pct  = train_raw.isnull().mean() * 100
cols_to_drop = missing_pct[missing_pct > 80].index.tolist()
train = train_raw.drop(columns=cols_to_drop)
del train_raw

# Drop identifier column
train = train.drop(columns=['TransactionID'], errors='ignore')

# Stratified 80/20 split
X = train.drop(columns=[TARGET])
y = train[TARGET]
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)
del train, X, y

print(f'✅ X_train shape : {X_train.shape}')
print(f'✅ X_val shape   : {X_val.shape}')
print(f'✅ Train fraud % : {y_train.mean()*100:.2f}%')
print(f'✅ Val fraud %   : {y_val.mean()*100:.2f}%')

---
## 🚀 Step 1 — Initialize LightGBM Model

In [ ]:
# Initialize LightGBM classifier with specified parameters
model = lgb.LGBMClassifier(
    objective='binary',
    metric='auc',
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=64,
    scale_pos_weight=27.6,
    random_state=42,
    verbose=-1
)

print('✅ LightGBM model initialized')
print(f'   Objective: binary')
print(f'   Metric: auc')
print(f'   Max estimators: 1000')
print(f'   Learning rate: 0.05')
print(f'   Num leaves: 64')
print(f'   Scale pos weight: 27.6')
print(f'   Early stopping rounds: 100')

---
## 🎯 Step 2 — Train Model with Early Stopping

In [ ]:
print('Training LightGBM model...')
print()

# Train with early stopping
model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric='auc',
    callbacks=[
        lgb.early_stopping(stopping_rounds=100, verbose=True),
        lgb.log_evaluation(period=50)
    ]
)

print()
print('✅ Training complete')

---
## 📊 Step 3 — Evaluate on Validation Set

In [ ]:
# Get predictions on validation set
y_val_pred_proba = model.predict_proba(X_val)[:, 1]

# Compute metrics
roc_auc = roc_auc_score(y_val, y_val_pred_proba)
pr_auc = average_precision_score(y_val, y_val_pred_proba)

# Get best iteration
best_iter = model.best_iteration_

# Print results
print('=' * 60)
print('   VALIDATION METRICS')
print('=' * 60)
print(f'Validation ROC-AUC:        {roc_auc:.6f}')
print(f'Validation PR-AUC:         {pr_auc:.6f}')
print(f'Best Iteration:            {best_iter}')
print('=' * 60)

---
## 🏆 Step 4 — Top 20 Feature Importances

In [ ]:
# Get feature importances
feature_importance = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': model.feature_importances_
}).sort_values('Importance', ascending=False).reset_index(drop=True)

feature_importance.index += 1
top20 = feature_importance.head(20)

# Print top 20 features
W = 70
sep = '=' * W
dash = '-' * W
hdr = f'{"Rank":<6} {"Feature":<30} {"Importance":>20}'

print(sep)
print('   TOP 20 FEATURES — By Importance (Gain)')
print(sep)
print(hdr)
print(dash)
for rank, row in top20.iterrows():
    print(
        f'{rank:<6} {row["Feature"]:<30} '
        f'{row["Importance"]:>20,.2f}'
    )
print(sep)